# Deepfake Detection with InceptionResNetV1

This notebook demonstrates the training process for a deepfake detection model using the InceptionResNetV1 architecture. The model is trained to classify facial images as either real or fake.

## Overview
- Data preparation and visualization
- Model architecture and configuration
- Training and validation pipeline
- Performance monitoring with Weights & Biases
- Model checkpointing


## 1. Setup and Imports

Import required libraries and set up environment variables.

In [ ]:
import os
import torch
import wandb
from torch import nn
from tqdm import tqdm
from torch import optim
from random import randint
import matplotlib.pyplot as plt
import torch.utils.data as data_utils
from facenet_pytorch import InceptionResnetV1

from dataset.faces import Faces
from utils.metrics import get_accuracy
from utils.dict_to_struct import DictStructure
from utils.denormalize_image import denormalize_image

## 2. Configuration

Set up constants and configuration parameters for the training process.

In [ ]:
# Dataset and hardware configuration
FACES_FOLDER = os.path.join("data", "faces")
FACES_CSV = os.path.join(FACES_FOLDER, "faces.csv")
VALIDATION_SPLIT = 0.2
CLASS_NAMES = {0: "real", 1: "fake"}
DEVICE = 'mps' if torch.mps.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

# Model checkpoint directory
MODEL_SAVE_DIR = "./models"
BASE_MODEL_PATH = os.path.join(MODEL_SAVE_DIR, "resnetinceptionv1")

In [ ]:
# Training hyperparameters
config = dict(
    epochs=80,
    save_every=4,
    classes=1,
    batch_size=32,
    learning_rate=0.001,
    dataset="FaceForensics++",
    architecture="InceptionResNetV1",
    pretrained_dataset="vggface2"
)

## 3. Data Preparation

Functions for creating and managing datasets and dataloaders.

In [ ]:
class DataManager:
    """Handles dataset creation, splitting, and dataloader configuration."""
    
    @staticmethod
    def get_data(split="training", transform=False, print_ds_len=False):
        """
        Create and split datasets for training and validation.
        
        Args:
            split (str): Either "training" or "validation"
            transform (bool): Whether to apply data transformations
            print_ds_len (bool): Whether to print dataset length info
            
        Returns:
            torch.utils.data.Subset: The requested dataset subset
        """
        assert split in ["training", "validation"], "Split must be 'training' or 'validation'"
        
        # Create the full dataset
        full_dataset = Faces(root=FACES_FOLDER, csv=FACES_CSV, split=split, transform=transform)
        if print_ds_len:
            print(f"Original dataset length: {len(full_dataset)}")

        # Calculate indices for the split
        end_idx = round((1 - VALIDATION_SPLIT) * len(full_dataset)) if split == 'training' else len(full_dataset)
        begin_idx = 0 if split == 'training' else round((1 - VALIDATION_SPLIT) * len(full_dataset))

        # Create the subset
        subset_indices = torch.arange(begin_idx, end_idx)
        subset_dataset = data_utils.Subset(full_dataset, subset_indices)

        # Print information about the subset
        split_percentage = round(len(subset_dataset) / len(full_dataset), 2) * 100
        print(f"{split.capitalize()} dataset size: {len(subset_dataset)} images ({split_percentage}% of total)")

        return subset_dataset
    
    @staticmethod
    def create_loader(dataset, batch_size, shuffle=True):
        """
        Create a DataLoader for the given dataset.
        
        Args:
            dataset: The dataset to create a loader for
            batch_size (int): Batch size for training
            shuffle (bool): Whether to shuffle the data
            
        Returns:
            torch.utils.data.DataLoader: Configured data loader
        """
        return data_utils.DataLoader(dataset=dataset, batch_size=batch_size, shuffle=shuffle, pin_memory=True, num_workers=2 )
    
    @staticmethod
    def visualize_samples(training_dataset, validation_dataset, num_samples=1):
        """
        Visualize sample images from both training and validation datasets.
        
        Args:
            training_dataset: Training dataset subset
            validation_dataset: Validation dataset subset
            num_samples (int): Number of sample pairs to display
        """
        for i in range(num_samples):
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
            
            # Get a random sample from each dataset
            train_idx = randint(0, len(training_dataset)-1)
            val_idx = randint(0, len(validation_dataset)-1)
            
            train_image, train_label = training_dataset[train_idx]
            val_image, val_label = validation_dataset[val_idx]
            
            # Denormalize images for visualization
            train_image = denormalize_image(train_image)
            val_image = denormalize_image(val_image)
            
            # Plot the images with their corresponding labels
            ax1.set_title(f"Training Sample ({CLASS_NAMES[train_label.int().item()]})")
            ax1.imshow(train_image)
            ax1.axis('off')
            
            ax2.set_title(f"Validation Sample ({CLASS_NAMES[val_label.int().item()]})")
            ax2.imshow(val_image)
            ax2.axis('off')
            
            plt.tight_layout()
            plt.show()

## 4. Model Setup

Functions for creating and initializing the model, loss function, and optimizer.

In [ ]:
class ModelBuilder:
    """Handles model creation and configuration."""
    
    @staticmethod
    def create_model_components(config):
        """
        Create and initialize model, dataloaders, loss function, and optimizer.
        
        Args:
            config: Configuration parameters (wandb.Config or dict)
            
        Returns:
            tuple: (model, train_loader, val_loader, criterion, optimizer)
        """
        # Ensure config is in the correct format
        if isinstance(config, wandb.Config):
            print("Using configuration from wandb.config")
        elif isinstance(config, dict):
            print("Converting dictionary configuration to struct")
            config = DictStructure(config)
        else:
            raise ValueError("config must be a wandb.Config or a dict")

        # Create datasets
        train_dataset = DataManager.get_data(split="training", transform=False, print_ds_len=True)
        val_dataset = DataManager.get_data(split="validation", transform=True, print_ds_len=True)
        
        # Create dataloaders
        train_loader = DataManager.create_loader(train_dataset, batch_size=config.batch_size)
        val_loader = DataManager.create_loader(val_dataset, batch_size=config.batch_size)
        
        # Initialize model
        model = InceptionResnetV1(pretrained=config.pretrained_dataset, classify=True, num_classes=config.classes, device=DEVICE)

        # Setup loss function and optimizer
        criterion = nn.BCEWithLogitsLoss()  # sigmoid + binary cross entropy
        optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)
        
        return model, train_loader, val_loader, criterion, optimizer
    
    @staticmethod
    def save_checkpoint(model, optimizer, loss, base_path, epoch):
        """
        Save model checkpoint.
        
        Args:
            model: PyTorch model to save
            optimizer: Optimizer state to save
            loss (float): Current validation loss
            base_path (str): Base path for saving the model
            epoch (int): Current epoch number
        """
        # Ensure the directory exists
        os.makedirs(os.path.dirname(base_path), exist_ok=True)
        
        # Save the checkpoint
        checkpoint_path = f"{base_path}_epoch_{epoch}.pth"
        torch.save({ 'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'validation_loss': loss,}, checkpoint_path)
        
        print(f"Model checkpoint saved to {checkpoint_path}")

## 5. Training Functions

Functions for processing batches and running the training loop.

In [ ]:
class Trainer:
    """Handles training and evaluation of the model."""
    
    @staticmethod
    def process_batch(images, labels, model, criterion, is_training=True, optimizer=None):
        """
        Process a single batch of data.
        
        Args:
            images: Batch of input images
            labels: Batch of labels
            model: PyTorch model
            criterion: Loss function
            is_training (bool): Whether in training mode
            optimizer: Optimizer for training
            
        Returns:
            tuple: (loss, accuracy)
        """
        # Validation for training mode
        if is_training and optimizer is None:
            raise ValueError("Optimizer must be provided for training")
            
        # Move tensors to device
        images = images.to(device=DEVICE)
        labels = labels.to(device=DEVICE)
        
        # Forward pass
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)
        accuracy = get_accuracy(outputs, labels)
        
        # Backward pass (only in training mode)
        if is_training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        return loss.item(), accuracy
    
    @staticmethod
    def train_model(model, train_loader, val_loader, criterion, optimizer, config):
        """
        Train the model for the specified number of epochs.
        
        Args:
            model: PyTorch model to train
            train_loader: Training data loader
            val_loader: Validation data loader
            criterion: Loss function
            optimizer: Optimizer
            config: Training configuration
            
        Returns:
            dict: Training history
        """
        print("\n=== Starting Full Model Training ===")
        
        # Set up wandb monitoring
        wandb.watch(model, criterion, log="all", log_freq=10)
        
        # Track training progress
        history = { "loss": {"train": [], "val": []}, "acc": {"train": [], "val": []} }
        batches_seen = 0
        
        # Main training loop
        for epoch in range(config.epochs):
            print(f"\nEpoch {epoch + 1}/{config.epochs}")
            
            # Training phase
            model.train()
            train_loss, train_acc = 0, 0
            print("Training:")
            for images_batch, labels_batch in tqdm(train_loader):
                batch_loss, batch_acc = Trainer.process_batch(images_batch, labels_batch, model, criterion, is_training=True, optimizer=optimizer)
                
                train_loss += batch_loss
                train_acc += batch_acc
                batches_seen += 1
            
            # Calculate average training metrics
            avg_train_loss = train_loss / len(train_loader)
            avg_train_acc = train_acc / len(train_loader)
            
            # Validation phase
            model.eval()
            val_loss, val_acc = 0, 0
            print("Validation:")
            with torch.no_grad():
                for images_batch, labels_batch in tqdm(val_loader):
                    batch_loss, batch_acc = Trainer.process_batch(images_batch, labels_batch, model, criterion, is_training=False)
                    
                    val_loss += batch_loss
                    val_acc += batch_acc
            
            # Calculate average validation metrics
            avg_val_loss = val_loss / len(val_loader)
            avg_val_acc = val_acc / len(val_loader)
            
            # Store metrics in history
            history["loss"]["train"].append(avg_train_loss)
            history["loss"]["val"].append(avg_val_loss)
            history["acc"]["train"].append(avg_train_acc)
            history["acc"]["val"].append(avg_val_acc)
            
            # Print epoch results
            print(f"Results - Train Loss: {avg_train_loss:.4f}, Train Acc: {avg_train_acc*100:.2f}%")
            print(f"          Val Loss: {avg_val_loss:.4f}, Val Acc: {avg_val_acc*100:.2f}%")
            
            # Log to wandb
            wandb.log({
                "Epoch": epoch + 1,
                "Train Loss": avg_train_loss,
                "Val Loss": avg_val_loss,
                "Train Acc": round(avg_train_acc * 100, 2),
                "Val Acc": round(avg_val_acc * 100, 2)
            }, step=batches_seen)
            
            # Save checkpoint if needed
            if (epoch + 1) % config.save_every == 0:
                ModelBuilder.save_checkpoint(
                    model, optimizer, avg_val_loss, 
                    BASE_MODEL_PATH, epoch + 1
                )
        
        print("\nTraining complete!")
        return history

## 6. Complete Training Pipeline

Combine all components into a complete training pipeline.

In [ ]:
def run_training_pipeline(hyperparameters):
    """
    Complete end-to-end training pipeline.
    
    Args:
        hyperparameters: Dictionary of hyperparameters
        
    Returns:
        model: Trained PyTorch model
    """
    # Initialize wandb
    with wandb.init(project="deepfake.ai", entity="jvishu-ai", config=hyperparameters):
        # Get configuration
        config = wandb.config
        
        # Build model and components
        print("Initializing model and components...")
        model, train_loader, val_loader, criterion, optimizer = ModelBuilder.create_model_components(config)
        
        # Visualize dataset samples
        print("\nVisualizing dataset samples:")
        train_dataset = train_loader.dataset
        val_dataset = val_loader.dataset
        DataManager.visualize_samples(train_dataset, val_dataset, num_samples=1)
        
        # Train the model
        Trainer.train_model(model, train_loader, val_loader, criterion, optimizer, config)
        
        return model

## 7. Execute Training

Initialize Weights & Biases and run the training pipeline.

In [ ]:
# Login to Weights & Biases
wandb.login()

In [ ]:
# Run the training pipeline
model = run_training_pipeline(config)

## 8. Model Evaluation and Usage

After training, you can use the model for inference on new images.

In [ ]:
def load_trained_model(checkpoint_path, device=DEVICE):
    """
    Load a trained model from a checkpoint.
    
    Args:
        checkpoint_path (str): Path to model checkpoint
        device: Device to load model on
        
    Returns:
        model: Loaded PyTorch model
    """
    # Initialize model
    model = InceptionResnetV1(pretrained=None, classify=True, num_classes=1, device=device)
    
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    print(f"Model loaded from {checkpoint_path}")
    print(f"Checkpoint from epoch {checkpoint['epoch']} with validation loss {checkpoint['validation_loss']:.4f}")
    
    return model

In [ ]:
def predict_image(model, image_tensor, threshold=0.5):
    """
    Predict whether an image is real or fake.
    
    Args:
        model: Trained PyTorch model
        image_tensor: Preprocessed image tensor
        threshold: Classification threshold
        
    Returns:
        tuple: (prediction, confidence)
    """
    # Ensure model is in evaluation mode
    model.eval()
    
    # Move image to device
    image_tensor = image_tensor.unsqueeze(0).to(DEVICE)
    
    # Get prediction
    with torch.no_grad():
        output = model(image_tensor).squeeze()
        
    # Apply sigmoid to get probability
    probability = torch.sigmoid(output).item()
    
    # Classify based on threshold
    prediction = 1 if probability >= threshold else 0
    
    return CLASS_NAMES[prediction], probability